# Lab 06 External V2 — 07 Validation

Final validation for the recurring Lab 06 Gold pipeline.

Validates:
- 6 dimensions
- 2 fact tables
- 4 aggregate tables
- key uniqueness / null checks
- fact grains
- foreign keys
- aggregate reconciliation
- basic business sanity

Governance setup, alert simulation, dashboard creation, and Genie creation remain outside the recurring Gold Job.

## 1. Runtime context

In [ ]:
import sys
from pathlib import Path

lab_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.runtime_config import load_runtime_context


ctx = load_runtime_context(
    dbutils,
    include_validation=True,
)

config = ctx.config
run_validation = ctx.run_validation

## 2. Shared configuration

In [ ]:
from pyspark.sql import functions as F
from src.external_tables import describe_table_location

print(f"Target schema : {config.target_schema_fqn}")
print(f"External root : {config.external_gold_root}")

## 3. Required Gold tables

In [ ]:
required_tables = {
    "dim_date": config.dim_date,
    "dim_patient": config.dim_patient,
    "dim_provider": config.dim_provider,
    "dim_organization": config.dim_organization,
    "dim_payer": config.dim_payer,
    "dim_condition": config.dim_condition,
    "fact_encounters": config.fact_encounters,
    "fact_conditions": config.fact_conditions,
    "agg_daily_encounters": config.agg_daily_encounters,
    "agg_organization_performance": config.agg_organization_performance,
    "agg_payer_performance": config.agg_payer_performance,
    "agg_condition_summary": config.agg_condition_summary,
}

table_rows = []
missing_tables = []

for name, table_name in required_tables.items():
    exists = spark.catalog.tableExists(table_name)
    table_rows.append((name, table_name, "PASS" if exists else "FAIL"))
    if not exists:
        missing_tables.append(table_name)

display(
    spark.createDataFrame(
        table_rows,
        ["object", "table_name", "status"],
    ).orderBy("object")
)

if missing_tables:
    raise RuntimeError("Missing Gold tables: " + ", ".join(missing_tables))

## 4. Inventory

In [ ]:
inventory = []

for name, table_name in required_tables.items():
    df = spark.table(table_name)
    inventory.append((name, df.count(), len(df.columns)))

display(
    spark.createDataFrame(
        inventory,
        ["object", "row_count", "column_count"],
    ).orderBy("object")
)

## 5. Dimension key validation

In [ ]:
dimension_keys = {
    "dim_date": (config.dim_date, "date_key"),
    "dim_patient": (config.dim_patient, "patient_key"),
    "dim_provider": (config.dim_provider, "provider_key"),
    "dim_organization": (config.dim_organization, "organization_key"),
    "dim_payer": (config.dim_payer, "payer_key"),
    "dim_condition": (config.dim_condition, "condition_key"),
}

dimension_failures = []
dimension_results = []

for name, (table_name, key_col) in dimension_keys.items():
    row = (
        spark.table(table_name)
        .agg(
            F.count("*").alias("rows"),
            F.countDistinct(key_col).alias("distinct_keys"),
            F.sum(F.when(F.col(key_col).isNull(), 1).otherwise(0)).alias("null_keys"),
        )
        .first()
    )

    passed = row["rows"] == row["distinct_keys"] and int(row["null_keys"] or 0) == 0

    dimension_results.append(
        (name, int(row["rows"]), int(row["distinct_keys"]), int(row["null_keys"] or 0),
         "PASS" if passed else "FAIL")
    )

    if not passed:
        dimension_failures.append(name)

display(
    spark.createDataFrame(
        dimension_results,
        ["dimension", "row_count", "distinct_keys", "null_keys", "status"],
    )
)

## 6. Fact grain validation

In [ ]:
fact_specs = {
    "fact_encounters": (config.fact_encounters, "encounter_key"),
    "fact_conditions": (config.fact_conditions, "condition_event_key"),
}

fact_failures = []
fact_results = []

for name, (table_name, key_col) in fact_specs.items():
    row = (
        spark.table(table_name)
        .agg(
            F.count("*").alias("rows"),
            F.countDistinct(key_col).alias("distinct_keys"),
            F.sum(F.when(F.col(key_col).isNull(), 1).otherwise(0)).alias("null_keys"),
        )
        .first()
    )

    passed = row["rows"] == row["distinct_keys"] and int(row["null_keys"] or 0) == 0

    fact_results.append(
        (name, int(row["rows"]), int(row["distinct_keys"]), int(row["null_keys"] or 0),
         "PASS" if passed else "FAIL")
    )

    if not passed:
        fact_failures.append(name)

display(
    spark.createDataFrame(
        fact_results,
        ["fact", "row_count", "distinct_grain_keys", "null_grain_keys", "status"],
    )
)

## 7. Foreign-key validation

In [ ]:
enc = spark.table(config.fact_encounters)
cond = spark.table(config.fact_conditions)

fk_checks = [
    ("encounter_missing_date_key", enc.filter(F.col("date_key").isNull()).count()),
    ("encounter_missing_patient_key", enc.filter(F.col("patient_key").isNull()).count()),
    ("encounter_missing_organization_key", enc.filter(F.col("organization_key").isNull()).count()),
    ("encounter_missing_payer_key", enc.filter(F.col("payer_key").isNull()).count()),
    ("encounter_unresolved_provider_key",
        enc.filter(F.col("provider_id").isNotNull() & F.col("provider_key").isNull()).count()),
    ("condition_missing_patient_key", cond.filter(F.col("patient_key").isNull()).count()),
    ("condition_missing_condition_key", cond.filter(F.col("condition_key").isNull()).count()),
    ("condition_missing_start_date_key", cond.filter(F.col("condition_start_date_key").isNull()).count()),
    ("condition_unresolved_encounter_key",
        cond.filter(F.col("encounter_id").isNotNull() & F.col("encounter_key").isNull()).count()),
]

fk_failures = [name for name, invalid_rows in fk_checks if invalid_rows != 0]

display(
    spark.createDataFrame(
        [(name, invalid_rows, "PASS" if invalid_rows == 0 else "FAIL")
         for name, invalid_rows in fk_checks],
        ["check_name", "invalid_rows", "status"],
    )
)

## 8. Aggregate grain validation

In [ ]:
aggregate_specs = {
    "agg_daily_encounters": (config.agg_daily_encounters, ["date_key"]),
    "agg_organization_performance": (config.agg_organization_performance, ["organization_key"]),
    "agg_payer_performance": (config.agg_payer_performance, ["payer_key"]),
    "agg_condition_summary": (config.agg_condition_summary, ["condition_key"]),
}

aggregate_failures = []
aggregate_results = []

for name, (table_name, grain_cols) in aggregate_specs.items():
    df = spark.table(table_name)

    duplicate_groups = (
        df.groupBy(*grain_cols)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_expr = " OR ".join(f"`{c}` IS NULL" for c in grain_cols)
    null_rows = df.filter(F.expr(null_expr)).count()

    passed = duplicate_groups == 0 and null_rows == 0

    aggregate_results.append(
        (name, duplicate_groups, null_rows, "PASS" if passed else "FAIL")
    )

    if not passed:
        aggregate_failures.append(name)

display(
    spark.createDataFrame(
        aggregate_results,
        ["aggregate", "duplicate_grain_groups", "null_grain_rows", "status"],
    )
)

## 9. Aggregate reconciliation

In [ ]:
fact_encounter_count = enc.count()
fact_condition_count = cond.count()

reconciliation = [
    (
        "agg_daily_encounters",
        fact_encounter_count,
        int(spark.table(config.agg_daily_encounters)
            .agg(F.sum("encounter_count").alias("v")).first()["v"] or 0),
    ),
    (
        "agg_organization_performance",
        fact_encounter_count,
        int(spark.table(config.agg_organization_performance)
            .agg(F.sum("encounter_count").alias("v")).first()["v"] or 0),
    ),
    (
        "agg_payer_performance",
        fact_encounter_count,
        int(spark.table(config.agg_payer_performance)
            .agg(F.sum("encounter_count").alias("v")).first()["v"] or 0),
    ),
    (
        "agg_condition_summary",
        fact_condition_count,
        int(spark.table(config.agg_condition_summary)
            .agg(F.sum("condition_event_count").alias("v")).first()["v"] or 0),
    ),
]

reconciliation_failures = [
    name for name, source_rows, aggregate_rows in reconciliation
    if source_rows != aggregate_rows
]

display(
    spark.createDataFrame(
        [
            (name, source_rows, aggregate_rows,
             "PASS" if source_rows == aggregate_rows else "FAIL")
            for name, source_rows, aggregate_rows in reconciliation
        ],
        ["aggregate", "source_fact_rows", "reconciled_rows", "status"],
    )
)

## 10. Business sanity checks

In [ ]:
business_checks = [
    ("fact_encounters_non_empty", fact_encounter_count > 0),
    ("fact_conditions_non_empty", fact_condition_count > 0),
    ("dim_date_non_empty", spark.table(config.dim_date).count() > 0),
    ("no_negative_claim_cost", enc.filter(F.col("total_claim_cost") < 0).count() == 0),
    ("no_negative_duration", enc.filter(F.col("duration_minutes") < 0).count() == 0),
]

business_failures = [name for name, passed in business_checks if not passed]

display(
    spark.createDataFrame(
        [(name, "PASS" if passed else "FAIL") for name, passed in business_checks],
        ["check_name", "status"],
    )
)

## 11. Operational artifact check

In [ ]:
operational_objects = {
    "RLS mapping table": config.table("lab06_user_organization_access"),
    "CLS mapping table": config.table("lab06_patient_data_privileged_users"),
    "Alert metrics table": config.table("lab06_data_volume_metrics"),
}

display(
    spark.createDataFrame(
        [
            (
                label,
                object_name,
                "AVAILABLE" if spark.catalog.tableExists(object_name) else "NOT CREATED",
            )
            for label, object_name in operational_objects.items()
        ],
        ["artifact", "object_name", "status"],
    )
)

## 12. Final validation summary

In [ ]:
final_checks = [
    ("required_tables", len(missing_tables) == 0),
    ("dimension_keys", len(dimension_failures) == 0),
    ("fact_grains", len(fact_failures) == 0),
    ("foreign_keys", len(fk_failures) == 0),
    ("aggregate_grains", len(aggregate_failures) == 0),
    ("aggregate_reconciliation", len(reconciliation_failures) == 0),
    ("business_sanity", len(business_failures) == 0),
]

failed_final_checks = [name for name, passed in final_checks if not passed]

display(
    spark.createDataFrame(
        [(name, "PASS" if passed else "FAIL") for name, passed in final_checks],
        ["validation_area", "status"],
    )
)

if run_validation and failed_final_checks:
    raise RuntimeError(
        "Lab 06 final validation failed: " + ", ".join(failed_final_checks)
    )

## External storage-location validation


In [ ]:
location_rows = []
location_failures = []

for logical_name, table_name in required_tables.items():
    expected = config.table_path(logical_name).rstrip("/")
    actual = describe_table_location(spark, table_name)

    status = "PASS" if actual == expected else "FAIL"
    location_rows.append(
        (logical_name, table_name, expected, actual, status)
    )

    if status == "FAIL":
        location_failures.append(logical_name)

display(
    spark.createDataFrame(
        location_rows,
        ["object", "table_name", "expected_path", "actual_path", "status"],
    ).orderBy("object")
)

if run_validation and location_failures:
    raise RuntimeError(
        "External location validation failed: "
        + ", ".join(location_failures)
    )

## 13. Completion

In [ ]:
print("LAB 06 — FINAL GOLD VALIDATION COMPLETE")
print("Recurring Gold pipeline status: PASS")
print("Validated: 6 dimensions, 2 facts, 4 aggregates, FKs, reconciliation, sanity checks")